<a href="https://colab.research.google.com/github/MangoCode13/PowerPINN/blob/main/PINN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Physics-Informed Neural Network (IEEE 14-Bus)
This notebook setup up the envirnment, power flow model, and PINN. Training data created separately is used to train the PINN.

# Clone Github Repo

In [ ]:
import os

# Clone the PowerPINN repository if it doesn't already exist
if not os.path.exists('PowerPINN'):
    !git clone https://github.com/MangoCode13/PowerPINN.git
    print("Repository 'PowerPINN' cloned successfully.")
else:
    print("Repository 'PowerPINN' already exists.")

## Download dependencies for environment
Use pip install during first use.

In [ ]:
!pip install -r ./PowerPINN/requirements.txt

## 1) Initialization

In [ ]:
import importlib
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pytorch_lightning as L

In [ ]:
# Use IEEE 14-bus network

import pandapower as pp
import pandapower.networks as pn

net = pn.case14() # Load your project's 14-bus system

pp.runpp(net)     # Run the power flow with Numba disabled

print("Model loaded and power flow executed successfully.")

## IEEE 14 Bus System
Single line diagram of system:

<img src="https://github.com/MangoCode13/PowerPINN/blob/main/images/IEEE_14_bus_System.png?raw=1" alt="IEEE 14 Bus System Diagram" width="400"/>

## Grid State: Evalutating the 14 Busy System with Powerpanda
Voltage magnitude and the phase angle

Role in PINN: Labels for the **Data Loss** term.

In [ ]:
print("=== STATE: net.res_bus ===")
state_cols = [c for c in ["vm_pu", "va_degree", "p_mw", "q_mvar"] if c in net.res_bus.columns]
print(net.res_bus[state_cols])

## Line Flows
Power P and Q

Role in PINN: Measurements the attacker manipulates.

In [ ]:
print("=== FLOWS: net.res_line ===")
flow_cols = [c for c in ["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar", "loading_percent"] if c in net.res_line.columns]
print(net.res_line[flow_cols])

## Balance Generation and System Power Transfer
Role in PINN: Used to verify Kirchhoff’s Current Law (KCL).

In [ ]:
print("=== BALANCE: net.res_gen ===")
if len(net.res_gen) > 0:
    print(net.res_gen[[c for c in ["p_mw", "q_mvar"] if c in net.res_gen.columns]])
else:
    print("No generator results available.")

print("\n=== BALANCE: net.res_ext_grid ===")
print(net.res_ext_grid[[c for c in ["p_mw", "q_mvar"] if c in net.res_ext_grid.columns]])

## Topology: Line Connections and Transformers
Role in PINN: Physical network constraints used by the **Physics Loss** term.

In [ ]:
print("=== TOPOLOGY: net.line ===")
line_cols = [c for c in ["from_bus", "to_bus", "length_km", "r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km"] if c in net.line.columns]
print(net.line[line_cols])

print("\n=== TOPOLOGY: net.trafo ===")
if len(net.trafo) > 0:
    trafo_cols = [c for c in ["hv_bus", "lv_bus", "sn_mva", "vk_percent", "vkr_percent"] if c in net.trafo.columns]
    print(net.trafo[trafo_cols])
else:
    print("No transformers in this network model.")

## Data Prep and Setup
Use Pandapower to create admittance matrix (Y_bus)

In [ ]:
# Convert Y-bus from pandapower to PyTorch tensors for PINN usage
def get_ybus_tensors(net: pp.pandapowerNet) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return Y-bus as real, imaginary, and complex Torch tensors for PINN usage."""
    # Access the internal admittance matrix assembled by pandapower.
    y_bus_complex = net._ppc["internal"]["Ybus"].todense()
    # Provide both split and complex forms for flexible downstream use.
    y_real = torch.tensor(np.real(y_bus_complex), dtype=torch.float32)
    y_imag = torch.tensor(np.imag(y_bus_complex), dtype=torch.float32)
    y_bus = torch.tensor(y_bus_complex, dtype=torch.complex64)
    return y_real, y_imag, y_bus

# Use function to extract admittance matrix tensor used by physics loss
_, _, Y_bus = get_ybus_tensors(net)

# Print summary of loaded data and Y-bus tensor
print(f'Extracted admittance matrix tensor successfully.')
print('Y_bus shape:', Y_bus.shape)

## Neural Network Architecture

In [ ]:
class PowerPINN(nn.Module):
    def __init__(self):
        super().__init__()
        # Width is 64 neurons and 4 hidden layers
        self.net = nn.Sequential(
            nn.Linear(28, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 28)
        )

    def forward(self, x):
        out = self.net(x)
        v_part = out[:, :14] + 1.0
        theta_part = out[:, 14:]
        return torch.cat([v_part, theta_part], dim=1)

# Using Stanford's ISLP Package to Train and View Our PINN Model.

 First we need to import ISLP modules. The SimpleDataModule and SimpleModule are simple versions of objects used in pytorch_lightning, the high-level module for fitting torch models. The ErrorTracker handles collections of targets and predictions over each mini-batch in the validation or test stage, allowing computation of the metric over the entire validation or test data set.

In [ ]:
from ISLP.torch import (SimpleDataModule,
                        SimpleModule,
                        ErrorTracker,
                        rec_num_workers)

# Load Training Data with PyTorch
Saves the resulting measurements (Inputs) and grid states (Labels) into a format ready for PINN training.

You can see the shapes of the input (P+Q measurements) and label (V+Theta states) tensors, along with the mean and standard deviation of the first five P-measurements, and the voltage and angle ranges.

In [ ]:
# Load the synthetically generated dataset
robust_data = torch.load('./PowerPINN/ieee14_training_data.pt')

# Extract tensors and rename for consistency with downstream cells
train_x = robust_data['train_x']
train_y = robust_data['train_y']
x_mean = robust_data['x_mean']
x_std = robust_data['x_std']

# Display summary information
print("Training data has been successfully loaded!")
print("--- Dataset Summary ---")
print(f"Input shape (X):  {train_x.shape} (Scenarios, P+Q measurements)")
print(f"Label shape (Y):  {train_y.shape} (Scenarios, V+Theta states)")
print(f"Mean of first 5 P-measurements: {x_mean[:5].numpy()}")
print(f"Std of first 5 P-measurements:  {x_std[:5].numpy()}")

# Quick check on label ranges
print(f"\nVoltage Range (V_pu): {train_y[:, :14].min():.4f} to {train_y[:, :14].max():.4f}")
print(f"Angle Range (rad):    {train_y[:, 14:].min():.4f} to {train_y[:, 14:].max():.4f}")

## Use SimpleDataModule to prepare training data.

Creates a TensorDataset: It wraps your train_x (input features) and train_y (labels) into a TensorDataset.

Determines number of workers: It uses rec_num_workers() from ISLP.torch to recommend an optimal number of worker processes for data loading, which can speed up training.

Initializes SimpleDataModule: This module is a simplified wrapper around pytorch_lightning.LightningDataModule and is responsible for managing the data loading process, including splitting the dataset into training and validation sets (20% for validation in this case) and preparing data loaders with a specified batch_size (256).

In [ ]:
from torch.utils.data import TensorDataset

# Create a TensorDataset from the newly normalized train_x and train_y
dataset = TensorDataset(train_x, train_y)

# Determine the number of workers for data loading
max_num_workers = rec_num_workers()

# Initialize SimpleDataModule with the created dataset
islp_dm = SimpleDataModule(dataset,
                            None, # test_dataset
                            validation=0.2,
                            num_workers=max_num_workers,
                            batch_size=256)

Taking a closer look at the synthetic training data set.

In [ ]:
from os import X_OK
for idx, (X, Y) in enumerate(islp_dm.train_dataloader()):
    print('X: ', train_x.shape)
    print('Y: ', train_y.shape)
    if idx >= 1:
      break

We see that for each batch there are 256 samples, `X` consists of 28 features (the 14 P measurements and 14 Q measurements). The network will output 28 'Y' values (14 voltage magnitudes and 14 phase angles).

Now we are ready to use the neural network we specified earlier with the physics loss function.

## Physics Loss Function
Define the `physics_loss` function to calculate the Kirchhoff's Current Law (KCL).


In [ ]:
import torch.nn as nn

def physics_loss(model, x_batch, Y_bus, x_mean, x_std):
    """Calculates the Kirchhoff's Current Law (KCL) mismatch as a physics-based loss."""

    # 1. Denormalize x_batch to get raw P and Q measurements
    # Add a small epsilon to x_std to prevent division by zero if std is 0
    x_raw = x_batch * (x_std + 1e-6) + x_mean

    # 2. Split into P and Q measurements
    p_measurements = x_raw[:, :14] # First 14 features are P
    q_measurements = x_raw[:, 14:] # Next 14 features are Q

    # 3. Use the model to predict voltage magnitudes and angles
    predicted_v_and_theta = model(x_batch)
    predicted_vm = predicted_v_and_theta[:, :14] # First 14 outputs are V_magnitude
    predicted_va = predicted_v_and_theta[:, 14:] # Next 14 outputs are V_angle

    # 4. Convert predicted voltage magnitudes and angles into complex voltages
    V_complex = predicted_vm * (torch.cos(predicted_va) + 1j * torch.sin(predicted_va))

    # 5. Calculate complex current I_complex = Y_bus * V_complex
    # Y_bus is (N, N), V_complex is (batch_size, N). For matmul, V_complex needs to be (N, batch_size)
    # and result will be (N, batch_size), then transpose back to (batch_size, N).
    I_complex = torch.matmul(Y_bus, V_complex.T).T

    # 6. Calculate apparent power S_complex = V_complex * I_complex*
    S_complex = V_complex * torch.conj(I_complex)

    # 7. Extract real part (P_calc) and imaginary part (Q_calc)
    P_calc = S_complex.real
    Q_calc = S_complex.imag

    # 8. Compute Mean Squared Error (MSE) between calculated and measured P and Q
    mse_p = nn.MSELoss()(P_calc, p_measurements)
    mse_q = nn.MSELoss()(Q_calc, q_measurements)

    # 9. Return the sum of these two MSEs as the physics_loss
    return mse_p + mse_q

## Training
Pytorch lightning is used to train the model.

α is use for the data loss weight
β is used for the physics loss weight.
α + β = 1

We use 100 epochs and a batch size of 256.

In [ ]:
# Get a single batch from the dataloader to check output dimensions
X_batch, Y_batch = next(iter(islp_dm.train_dataloader()))

# Run the batch through the PINN model
output_shape = model(X_batch).size()
print(f"Input shape: {X_batch.shape}")
print(f"Output shape: {output_shape}")

In [ ]:
from pytorch_lightning.loggers import CSVLogger

class LitPowerPINN(L.LightningModule):
    def __init__(self, model, Y_bus, x_mean, x_std, alpha=0.7):
        super().__init__()
        self.model = model
        self.Y_bus = Y_bus
        self.x_mean = x_mean
        self.x_std = x_std
        self.alpha = alpha # Data weight
        self.beta = 1.0 - alpha # Physics weight (Sum to 1)
        self.criterion = nn.MSELoss()

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self.model(x)
        loss_data = self.criterion(pred, y)
        loss_phys = physics_loss(self.model, x, self.Y_bus, self.x_mean, self.x_std)

        # Use normalized weights as per the paper
        total_loss = (self.alpha * loss_data) + (self.beta * loss_phys)

        self.log('train_loss', total_loss, on_epoch=True, prog_bar=True)
        self.log('data_loss', loss_data, on_epoch=True)
        self.log('physics_loss', loss_phys, on_epoch=True)
        return total_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

# Re-initialize with 70% Data, 30% Physics weighting
model = PowerPINN()
pinn_lit_model = LitPowerPINN(model, Y_bus, x_mean, x_std, alpha=0.7)
logger_pinn = CSVLogger("lightning_logs", name="normalized_weight_pinn")
trainer_pinn = L.Trainer(max_epochs=100, logger=logger_pinn, accelerator='auto', devices=1)
trainer_pinn.fit(pinn_lit_model, datamodule=islp_dm)

# Torchinfo
There are several other helper packages for `torch`. For instance,
the `torchmetrics` package has utilities to compute
various metrics to evaluate performance when fitting
a model. The `torchinfo` package provides a useful
summary of the layers of a model.


In [ ]:
from torchinfo import summary

In [ ]:
summary(model, input_size=train_x.shape, col_names=['input_size', 'output_size', 'num_params'])

# Chart of Data and Physics Loss

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

log_path = os.path.join(trainer_pinn.logger.log_dir, 'metrics.csv')
if os.path.exists(log_path):
    df_full = pd.read_csv(log_path)
    # Filter for rows that actually contain training metrics
    # Use 'data_loss_epoch' and 'physics_loss_epoch' as observed in the kernel state
    df = df_full.dropna(subset=['data_loss_epoch']) if 'data_loss_epoch' in df_full.columns else df_full

    plt.figure(figsize=(10, 6))
    if 'data_loss_epoch' in df.columns:
        plt.plot(df['epoch'], df['data_loss_epoch'], label='Data Loss (Supervised)')
    if 'physics_loss_epoch' in df.columns:
        plt.plot(df['epoch'], df['physics_loss_epoch'], label='Physics Loss (KCL Mismatch)')
    # Add Total Loss plot
    if 'train_loss_epoch' in df.columns:
        plt.plot(df['epoch'], df['train_loss_epoch'], label='Total Loss', linestyle='--')

    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss Value')
    plt.title('Comparison of Data, Physics, and Total Loss Components (High Capacity Model)')
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.show()

    print("Available columns:", df_full.columns.tolist())
else:
    print("Log not found yet. Please wait for training to complete.")

# Adversary Scenario: False Data Injection (FDI) Attack

In this section, we simulate a malicious cyber-attack where an adversary manipulates the sensor readings at Bus 4. We compare our PINN's performance against a standard Neural Network to demonstrate how physical constraints can detect and mitigate data manipulation.

### Evaluating Attack Resilience: PINN vs. Standard NN

To demonstrate the value of physics-informed training, we compared our **PowerPINN** against a **Data-Only NN** of the same architecture.

*   **The Attack**: We injected a malicious bias (FDI) into the power measurements of Bus 4.
*   **The Detection Mechanism**: We calculate the **Physics Residual**, which measures how much the predicted grid state violates Kirchhoff's Current Law ($I = YV$).
*   **The Result**: The standard NN produces a state that matches the manipulated data perfectly but results in a massive physics residual. The PINN, however, produces a state with a much lower residual, effectively 'rejecting' the non-physical manipulation and providing a clear signal for attack detection.

# Data Only Model Trained
This used to compare with the PINN

In [ ]:
class DataOnlyNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Same architecture as our PowerPINN for a fair comparison
        self.net = nn.Sequential(
            nn.Linear(28, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 28)
        )
    def forward(self, x):
        out = self.net(x)
        return torch.cat([out[:, :14] + 1.0, out[:, 14:]], dim=1)

# Quick training of the Data-Only model (Supervised loss only)
model_data_only = DataOnlyNN()
opt_std = torch.optim.Adam(model_data_only.parameters(), lr=1e-4)
criterion = nn.MSELoss()

print("Training Data-Only Baseline...")
for epoch in range(500):
    opt_std.zero_grad()
    p = model_data_only(train_x)
    loss = criterion(p, train_y)
    loss.backward()
    opt_std.step()
print("Baseline trained.")

## Simulated Attack Scenario
The attack manipuates P and Q data readings at Bus 4

In [ ]:
def simulate_pq_fdi_attack(input_batch, target_bus_idx=3, p_bias=1.5, q_bias=0.8):
    """Injects malicious bias into BOTH P and Q measurements of a bus."""
    attacked_input = input_batch.clone()
    # P is in first 14, Q is in last 14
    attacked_input[:, target_bus_idx] += p_bias
    attacked_input[:, target_bus_idx + 14] += q_bias
    return attacked_input

# 1. Generate the P+Q Attack
X_clean, Y_true = next(iter(islp_dm.train_dataloader()))
X_attacked_pq = simulate_pq_fdi_attack(X_clean, target_bus_idx=3, p_bias=1.5, q_bias=1.0)

# 2. Run Inference
model.eval()
model_data_only.eval()
with torch.no_grad():
    pred_pinn = model(X_attacked_pq)
    pred_std = model_data_only(X_attacked_pq)

    # 3. Calculate Physics Residuals (KCL Mismatch)
    res_pinn = physics_loss(model, X_attacked_pq, Y_bus, x_mean, x_std)
    res_std = physics_loss(model_data_only, X_attacked_pq, Y_bus, x_mean, x_std)

print(f"--- FDI Attack (P+Q Manipulation) Results ---")
print(f"Target: Bus 4 (Index 3)")
print(f"Standard NN Physics Residual: {res_std.item():.2f} (Massive outlier)")
print(f"PowerPINN Physics Residual:   {res_pinn.item():.2f} (Tighter clustering)")

improvement = (res_std.item() - res_pinn.item()) / res_std.item() * 100
print(f"\nVERDICT: The PINN is {improvement:.2f}% more consistent with grid physics.")
print("The Standard NN produces a 'physically impossible' state, while the PINN flags the attack.")

## Training Data Overview

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# train_x in this notebook is normalized
x_np = train_x.detach().cpu().numpy()
num_samples, num_features = x_np.shape
print(f"train_x shape: {x_np.shape} (samples x features)")
print("Feature layout: first 14 = P, last 14 = Q")

# 1) Heatmap of first N samples (normalized values)
n_show = min(120, num_samples)
plt.figure(figsize=(12, 4))
plt.imshow(x_np[:n_show], aspect='auto', cmap='coolwarm')
plt.colorbar(label='Normalized value')
plt.xlabel('Feature index (0-27)')
plt.ylabel('Sample index')
plt.title(f'Normalized Training Inputs (First {n_show} Samples)')
plt.axvline(13.5, color='black', linestyle='--', linewidth=1)
plt.text(4, -4, 'P features', fontsize=10)
plt.text(18, -4, 'Q features', fontsize=10)
plt.tight_layout()
plt.show()

# 2) Distribution summary for P and Q groups
p_vals = x_np[:, :14].reshape(-1)
q_vals = x_np[:, 14:].reshape(-1)

plt.figure(figsize=(10, 4))
plt.hist(p_vals, bins=50, alpha=0.7, label='P features (normalized)')
plt.hist(q_vals, bins=50, alpha=0.7, label='Q features (normalized)')
plt.xlabel('Normalized feature value')
plt.ylabel('Count')
plt.title('Feature Distributions: P vs Q')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 3) One scenario profile in original units (undo normalization)
sample_idx = 0
sample_norm = train_x[sample_idx]
sample_raw = (sample_norm * (x_std + 1e-6) + x_mean).detach().cpu().numpy()
p_sample = sample_raw[:14]
q_sample = sample_raw[14:]
bus = np.arange(1, 15)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(bus, p_sample)
ax[0].set_title(f'Sample {sample_idx}: Active Power P by Bus')
ax[0].set_xlabel('Bus')
ax[0].set_ylabel('P (scaled units)')
ax[0].grid(alpha=0.3)

ax[1].bar(bus, q_sample, color='tab:orange')
ax[1].set_title(f'Sample {sample_idx}: Reactive Power Q by Bus')
ax[1].set_xlabel('Bus')
ax[1].set_ylabel('Q (scaled units)')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Save Model for Evaluation

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'x_mean': x_mean,
    'x_std': x_std,
    'input_dim': 28,
    'output_dim': 28
}

torch.save(checkpoint, 'XXMONTH_pinn_model.pth')

print("Model saved successfully")